# 00 · Catalog & Environment Setup
### Creates catalog, schemas and all Delta tables
**Run this notebook once before anything else.**

In [0]:
%run ./config/pipeline_config

In [0]:
# ── Create Catalog ───────────────────────────────────────────────────
spark.sql(f"""
        CREATE CATALOG IF NOT EXISTS {CATALOG}
          COMMENT 'Weather pipeline catalog — powered by Open-Meteo API'
          """)
spark.sql(f"USE CATALOG {CATALOG}")
print(f"✅ Catalog '{CATALOG}' created successfully!")


In [0]:
# ── Create Bronze, Silver, Gold Schemas ──────────────────────────────
schemas={
    BRONZE : "Raw ingestion layer — unmodified API responses, append only",
    SILVER : "Cleaned and validated layer — flattened, deduped, merged",
    GOLD   : "Aggregated business metrics — ready for dashboards"

}

for schema, comment in schemas.items():
    spark.sql(f"""
              CREATE SCHEMA IF NOT EXISTS {schema}
              COMMENT '{comment}'
              """)
    print(f"✅ Schema '{CATALOG}.{schema}' created!")


In [0]:
# ── Bronze: ingestion_watermark ──────────────────────────────────────

spark.sql(f"""
          
        CREATE TABLE IF NOT EXISTS {TBL_WATERMARK} (
              
              source_name STRING NOT NULL,
              location_id INT NOT NULL,
              last_loaded_date INT NOT NULL,
              updated_at TIMESTAMP
          )
          USING DELTA
          COMMENT 'Tracks incremental load watermarks per source and location'
          TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')
          
          
          """)

print(f"✅ Table '{TBL_WATERMARK}' created!")

In [0]:
# ── Bronze: locations ────────────────────────────────────────────────


spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {TBL_LOCATIONS_RAW}(
        location_id   INT,
        city          STRING,
        country       STRING,
        latitude      DOUBLE,
        longitude     DOUBLE,
        timezone      STRING,
        climate_zone  STRING

    )
    USING DELTA
    COMMENT 'Seed table — cities we monitor, loaded once'
    TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')
    
    """
)

print(f"✅ Table '{TBL_LOCATIONS_RAW}' created!")


In [0]:
# ── Bronze: weather_hourly_raw ───────────────────────────────────────

spark.sql(f"""
          CREATE TABLE IF NOT EXISTS {TBL_WEATHER_RAW}(
        location_id        INT,
        city               STRING,
        observation_ts     TIMESTAMP,
        temperature_2m     DOUBLE,
        relative_humidity  DOUBLE,
        precipitation      DOUBLE,
        wind_speed_10m     DOUBLE,
        wind_direction_10m DOUBLE,
        surface_pressure   DOUBLE,
        cloud_cover        DOUBLE,
        weather_code       INT,
        _ingested_at       TIMESTAMP,
        _source_url        STRING,
        _batch_date        DATE 
          )
          USING DELTA 
          PARTITIONED BY (_batch_date)
          COMMENT 'Raw hourly weather data from Open-Meteo — append only'
          TBLPROPERTIES(
              'delta.enableChangeDataFeed' = 'true',
              'delta.autoOptimize.optimizeWrite' = 'true',
              'delta.autoOptimize.autoCompact' = 'true'
          )
          
          
          """)
print(f"✅ Table '{TBL_WEATHER_RAW}' created!")


In [0]:
# ── Bronze: air_quality_hourly_raw ───────────────────────────────────

spark.sql(f"""
          CREATE TABLE IF NOT EXISTS {TBL_AIR_QUALITY_RAW} (
            location_id        INT,
            city               STRING,
            observation_ts     TIMESTAMP,
            pm2_5              DOUBLE,
            pm10               DOUBLE,
            carbon_monoxide    DOUBLE,
            nitrogen_dioxide   DOUBLE,
            ozone              DOUBLE,
            dust               DOUBLE,
            european_aqi       INT,
            us_aqi             INT,
            _ingested_at       TIMESTAMP,
            _source_url        STRING,
            _batch_date        DATE
          )
          USING DELTA
          PARTITIONED BY (_batch_date)
    COMMENT 'Raw hourly air quality data from Open-Meteo — append only'
          TBLPROPERTIES(
              'delta.enableChangeDataFeed'='true',
              'delta.autoOptimize.optimizeWrite'='true',
              'delta.autoOptimize.autoCompact'='true'
          )

          """)
print(f"✅ Table '{TBL_AIR_QUALITY_RAW}' created!")



In [0]:

# ── Silver: dim_locations ────────────────────────────────────────────

spark.sql(f"""
          CREATE TABLE IF NOT EXISTS {TBL_DIM_LOCATIONS} (
            location_id   INT,
            city          STRING,
            country       STRING,
            latitude      DOUBLE,
            longitude     DOUBLE,
            timezone      STRING,
            climate_zone  STRING,
            _updated_at   TIMESTAMP
          ) 
          USING DELTA
          COMMENT 'Cleaned locations dimension table'
          TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')
          
          """)
print(f"✅ Table '{TBL_DIM_LOCATIONS}' created!")


# ── Silver: fact_weather_hourly ──────────────────────────────────────
spark.sql(f"""
          CREATE TABLE IF NOT EXISTS {TBL_FACT_WEATHER}(
            location_id        INT,
            city               STRING,
            observation_ts     TIMESTAMP,
            temperature_2m     DOUBLE,
            relative_humidity  DOUBLE,
            precipitation      DOUBLE,
            wind_speed_10m     DOUBLE,
            wind_direction_10m DOUBLE,
            surface_pressure   DOUBLE,
            cloud_cover        DOUBLE,
            weather_code       INT,
            weather_description STRING,
            is_rainy           BOOLEAN,
            is_stormy          BOOLEAN,
            _updated_at        TIMESTAMP

          )
          USING DELTA
          PARTITIONED BY (city)
          COMMENT 'Cleaned hourly weather fact table — merged incrementally'
          TBLPROPERTIES (
              'delta.enableChangeDataFeed' = 'true',
              'delta.autoOptimize.optimizeWrite' = 'true',
              'delta.autoOptimize.autoCompact' = 'true'
          )
          """)
print(f"✅ Table '{TBL_FACT_WEATHER}' created!")

# ── Silver: fact_air_quality_hourly ──────────────────────────────────

spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {TBL_FACT_AIR_QUALITY} (
            location_id      INT,
            city             STRING,
            observation_ts   TIMESTAMP,
            pm2_5            DOUBLE,
            pm10             DOUBLE,
            carbon_monoxide  DOUBLE,
            nitrogen_dioxide DOUBLE,
            ozone            DOUBLE,
            dust             DOUBLE,
            european_aqi     INT,
            us_aqi           INT,
            aqi_category     STRING,
            _updated_at      TIMESTAMP

          )

        USING DELTA
        PARTITIONED BY (city)
        COMMENT 'Cleaned hourly air quality fact table — merged incrementally'
        TBLPROPERTIES (
            'delta.enableChangeDataFeed' = 'true',
            'delta.autoOptimize.optimizeWrite' = 'true',
            'delta.autoOptimize.autoCompact' = 'true'
        )

          """)
print(f"✅ Table '{TBL_FACT_AIR_QUALITY}' created!")



In [0]:
# ── Silver: dq_results ───────────────────────────────────────────────
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {TBL_DQ_RESULTS}(
        run_id           STRING,
        run_ts           TIMESTAMP,
        source_table     STRING,
        check_name       STRING,
        status           STRING,
        records_checked  BIGINT,
        records_failed   BIGINT,
        failure_rate_pct DOUBLE,
        details          STRING
    )
    USING DELTA
    COMMENT 'Data quality check results log — written by quality notebook'


""")
print(f"✅ Table '{TBL_DQ_RESULTS}' created!")


In [0]:
# ── Gold: gold_daily_summary ─────────────────────────────────────────
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {TBL_GOLD_DAILY_SUMMARY} (
        location_id        INT,
        city               STRING,
        country            STRING,
        climate_zone       STRING,
        date               DATE,
        avg_temperature    DOUBLE,
        min_temperature    DOUBLE,
        max_temperature    DOUBLE,
        avg_humidity       DOUBLE,
        total_precipitation DOUBLE,
        avg_wind_speed     DOUBLE,
        avg_cloud_cover    DOUBLE,
        rainy_hours        INT,
        stormy_hours       INT,
        _updated_at        TIMESTAMP
    )
    USING DELTA
    PARTITIONED BY (city)
    COMMENT 'Daily weather summary per city — aggregated from silver'
    TBLPROPERTIES (
        'delta.autoOptimize.optimizeWrite' = 'true',
        'delta.autoOptimize.autoCompact'   = 'true'
    )
""")
print(f"✅ Table '{TBL_GOLD_DAILY_SUMMARY}' created!")

print(f"✅ Table '{TBL_GOLD_DAILY_SUMMARY}' created!")

# ── Gold: gold_city_comparison ───────────────────────────────────────


spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {TBL_GOLD_CITY_COMPARE}(
                date                   DATE,
                city                   STRING,
                country                STRING,
                climate_zone           STRING,
                avg_temperature        DOUBLE,
                total_precipitation    DOUBLE,
                avg_humidity           DOUBLE,
                avg_wind_speed         DOUBLE,
                temperature_rank       INT,
                precipitation_rank     INT,
                _updated_at            TIMESTAMP
          )
        USING DELTA
        COMMENT 'City vs city daily comparison — ranked metrics'
        TBLPROPERTIES (
            'delta.enableChangeDataFeed' = 'true',
            'delta.autoOptimize.optimizeWrite' = 'true'
        )
""")
print(f"✅ Table '{TBL_GOLD_CITY_COMPARE}' created!")


# ── Gold: gold_aqi_daily_index ───────────────────────────────────────

spark.sql(f"""
  CREATE TABLE IF NOT EXISTS {TBL_GOLD_AQI_DAILY}(
        location_id       INT,
        city              STRING,
        country           STRING,
        date              DATE,
        avg_pm2_5         DOUBLE,
        avg_pm10          DOUBLE,
        avg_ozone         DOUBLE,
        avg_nitrogen_dioxide DOUBLE,
        avg_european_aqi  DOUBLE,
        avg_us_aqi        DOUBLE,
        max_european_aqi  INT,
        max_us_aqi        INT,
        aqi_category      STRING,
        unhealthy_hours   INT,
        _updated_at       TIMESTAMP
  )    
  USING DELTA
  PARTITIONED BY (city)
  COMMENT 'Daily AQI summary per city — aggregated from silver'
  TBLPROPERTIES (
      'delta.enableChangeDataFeed' = 'true',
      'delta.autoOptimize.optimizeWrite' = 'true'
      )
          
""")
print(f"✅ Table '{TBL_GOLD_AQI_DAILY}' created!")


In [0]:
print("="*55)
print("      CATALOG STRUCTURE VERIFICATION")
print("="*55)

for schema in [BRONZE,SILVER,GOLD]:
    tables = spark.sql(f"SHOW TABLES IN {CATALOG}.{schema}").collect()
    print(f"\n {CATALOG}.{schema}/")
    for t in tables:
        print(f"    └── {t['tableName']}")
    
print("\n✅ All tables created successfully!")
print("✅ 00_catalog_setup complete!")
print("\n── Next Step ─────────────────────────────────────")
print("   Open bronze/01_bronze_ingestion")
